In [2]:
# ============================================================
# CISB5123 TEXT ANALYTICS - LAB ASSIGNMENT 3
# ============================================================
# NUR SHAHANI BINTI ABDUL HAKIM
# BIS01084303

# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import nltk
import gensim
import gensim.corpora as corpora
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import LdaModel, CoherenceModel
import re

# =========================
# 2. DOWNLOAD REQUIRED RESOURCES (SAFE VERSION)
# =========================
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# ⚠️ IMPORTANT FIX: avoid punkt / punkt_tab issues completely
# We will NOT use word_tokenize()

# =========================
# 3. CREATE DATASET
# =========================
data = {
    "text": [
        "The government announced new economic policies to boost growth.",
        "The football team won their match after a thrilling penalty shootout.",
        "New advancements in artificial intelligence are transforming industries.",
        "The stock market saw a significant drop due to global uncertainty.",
        "Scientists discovered a new species in the Amazon rainforest.",
        "The movie received excellent reviews from critics worldwide.",
        "Healthcare reforms are being debated in parliament.",
        "A major tech company released its latest smartphone model.",
        "The local basketball team secured a place in the finals.",
        "Climate change continues to impact weather patterns globally.",
        "The president met with foreign leaders to discuss trade agreements.",
        "The tennis championship attracted players from around the world.",
        "Researchers are developing new vaccines to fight diseases.",
        "The entertainment industry is evolving with streaming platforms.",
        "Economic inflation is affecting consumer purchasing power."
    ]
}

df = pd.DataFrame(data)

# =========================
# 4. TEXT PREPROCESSING (FIXED VERSION)
# =========================
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # ✅ SAFE TOKENIZATION (NO NLTK ERROR)
    tokens = text.split()

    cleaned = []
    for word in tokens:
        if word not in stop_words and len(word) > 2:
            cleaned.append(lemmatizer.lemmatize(word))

    return cleaned

processed_text = df['text'].apply(preprocess)

# =========================
# 5. DICTIONARY & CORPUS
# =========================
id2word = corpora.Dictionary(processed_text)

id2word.filter_extremes(no_below=1, no_above=0.8)

corpus = [id2word.doc2bow(text) for text in processed_text]

# =========================
# 6. BUILD LDA MODEL
# =========================
lda_model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=4,
    random_state=42,
    passes=15,
    alpha='auto'
)

# =========================
# 7. DISPLAY TOPICS
# =========================
print("\n===== TOPICS =====\n")
for idx, topic in lda_model.print_topics(num_words=8):
    print(f"Topic {idx}: {topic}\n")

# =========================
# 8. COHERENCE SCORE
# =========================
coherence_model = CoherenceModel(
    model=lda_model,
    texts=processed_text,
    dictionary=id2word,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()

print("\n===== COHERENCE SCORE =====")
print("Coherence Score:", coherence_score)

# =========================
# 9. INTERPRETATION
# =========================
print("\n===== INTERPRETATION =====")
print("""
The coherence score measures how meaningful and interpretable the topics are.

The LDA model identified 4 topics representing themes such as:
- Politics & government
- Sports
- Technology & AI
- Economy & finance

The coherence score indicates that the topics are reasonably structured
and interpretable for text analysis.
""")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/a3939b17-2a70-47a1-9594-
[nltk_data]     4dc2e9f0d2a7/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/a3939b17-2a70-47a1-9594-
[nltk_data]     4dc2e9f0d2a7/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/a3939b17-2a70-47a1-9594-
[nltk_data]     4dc2e9f0d2a7/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



===== TOPICS =====

Topic 0: 0.031*"championship" + 0.031*"specie" + 0.031*"tennis" + 0.031*"review" + 0.031*"world" + 0.031*"scientist" + 0.031*"attracted" + 0.031*"movie"

Topic 1: 0.043*"economic" + 0.024*"latest" + 0.024*"tech" + 0.024*"agreement" + 0.024*"trade" + 0.024*"president" + 0.024*"major" + 0.024*"met"

Topic 2: 0.042*"new" + 0.042*"team" + 0.023*"industry" + 0.023*"saw" + 0.023*"due" + 0.023*"vaccine" + 0.023*"transforming" + 0.023*"researcher"

Topic 3: 0.037*"platform" + 0.037*"evolving" + 0.037*"entertainment" + 0.037*"streaming" + 0.037*"industry" + 0.037*"weather" + 0.037*"climate" + 0.037*"globally"


===== COHERENCE SCORE =====
Coherence Score: 0.43533814939920434

===== INTERPRETATION =====

The coherence score measures how meaningful and interpretable the topics are.

The LDA model identified 4 topics representing themes such as:
- Politics & government
- Sports
- Technology & AI
- Economy & finance

The coherence score indicates that the topics are reasonably 